# Lab 2: Leaky Integrate-and-Fire Neurons
**CSE5026 · Introduction and Frontiers of Cognitive Science · Fall 2026**

Use the **Python (cogsci)** kernel. This notebook requires NumPy and Matplotlib.

## Task 1. Implementing Firing in LIF Neurons

In **Task 1A**, complete the three `TODO` expressions in the final `FirstOrderLIF` class and run the first simulation. In **Task 1B**, implement a simulation loop, count spikes, and compare firing rates across input currents and refractory periods. The second original simulation is optional parameter exploration. The earlier classes illustrate how the model develops; their `...` methods are explanatory placeholders, not additional coding tasks.

The code is adapted from [this tutorial](https://soney.github.io/snn-from-scratch/chapters/05%20-%20Implementing%20Firing.html).

After implementing the neuron, complete **Task 2: read and explain your first output figure**. No additional algorithm implementation is required for Task 2.

### Task 1A. Implement the neuron


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

Starting with the base class `FirstOrderLI`, we are going to incrementally add features to its `__init__()` and `step()` methods.

First, the very basic `FirstOrderLI` class is shown as follows:

In [ ]:
class FirstOrderLI: # First Order Leaky Integrate
    def __init__(self, tau_rc=0.2, v_init=0): # Default values for tau_rc and v_init
        self.tau_rc = tau_rc # Set instance variables
        self.v      = v_init

    def step(self, I, t_step): # Advance one time step (input I and time step size t_step)
        ...



In order to make a LIF neuron that is able to fire, we need to add two additional parameters:

- A threshold potential, $v_{th}$ where if the neuron's potential reaches this threshold, it will fire
- A refractory period, $\tau_{ref}$---after our neuron fires, this specifies how long until it will "accept" input again.

We are going to represent firing as the neuron's "output". By default, the output will be `0`. If a neuron is firing, its output will be `1/T_step` (multiplying this output by `T_step` gives a unit-area spike; this is a numerical representation, not a measure of physical energy).

We will thus add some additional instance variables to represent neuron parameters:
- `self.v_th` to represent the firing threshold, $v_{th}$. Default: `1`
- `self.tau_ref` to represent the refractory period, $\tau_{ref}$. Default: `0.002`

We also need to track two more instance variables:
- `self.output` to represent the output (`0` if not firing, `1/T_step` if firing at the current step)
- `self.refractory_time` to represent how much time we have left until our neuron can accept input.

In [ ]:
class FirstOrderLIF:
    def __init__(self, tau_rc=0.2, tau_ref=0.002, v_init=0, v_th=1): # ADDED: tau_ref and v_th
        self.tau_rc  = tau_rc
        self.v       = v_init

        # vvv ADDED vvv
        self.v_th    = v_th
        self.tau_ref = tau_ref

        self.output          = 0
        self.refractory_time = 0
        # ^^^ ADDED ^^^

    def step(self, I, t_step):
        ...

Then, we need to modify the `step(I, t_step)` function to add firing. 

First, after the voltage is updated, we need to check if it is greater than the threshold. If it is, we fire (set `self.output` to `1/t_step`) and reset the voltage (set `self.v` to `0`):

```python
if self.v >= self.v_th:      # Voltage is above the threshold
    self.output = 1 / t_step # Fire
    self.v = 0               # Reset potential
else:
    self.output = 0          # Don't fire
```

---

But we don't just want to set `self.v` to `0`. We want it to **stay** at `0` for $\tau_{ref}$ (stored in the instance variable `self.tau_ref`) seconds. 

So we use `self.refractory_time` to track how much time we have left until the neuron accepts input again, and modify the above code to set the refractory time remaining if the neuron fires (like a count-down timer):

```python
if self.v >= self.v_th:      # Voltage is above the threshold
    # vvv ADDED vvv
    self.refractory_time = self.tau_ref
    # ^^^ ADDED ^^^

    self.output = 1 / t_step # Fire
    self.v = 0               # Reset potential
else:
    self.output = 0          # Don't fire
```

---

The key is to update `self.refractory_time` every time we take a step:
```python
def step(self, I, t_step):
    self.refractory_time -= t_step # Subtract the amount of time that passed from our refractory time
    # ...
```

But we also need to check if we are in a refractory period before we update the voltage, so we need to wrap the update to `self.v` inside a conditional:

```python
def step(self, I, t_step):
    self.refractory_time -= t_step # Subtract the amount of time that passed from our refractory time

    if self.refractory_time < 0: # If we aren't in our refractory period, update the voltage
        ...
    
    # ...
```
---

Overall, this gives the code skeleton as follows, and you need to fill up the part for integrating the input current:

$v[t] = v[t-1] (1 - \frac{T_{step}}{\tau_{rc}}) + \frac{T_{step}}{\tau_{rc}}I[t]$

In [ ]:
class FirstOrderLIF: # First Order Leaky Integrate and Fire
    def __init__(self, tau_rc=0.2, tau_ref=0.002, v_init=0, v_th=1):
        self.tau_rc  = tau_rc  # Potential decay time constant
        self.v       = v_init  # Potential value
        self.v_th    = v_th    # Firing threshold
        self.tau_ref = tau_ref # Refractory period time constant

        self.output          = 0 # Current output value
        self.refractory_time = 0 # Current refractory period time (how long until the neuron can fire again)
    
    def step(self, I, t_step): # Advance one time step (input I and time step size t_step)
        self.refractory_time -= t_step # Subtract the amount of time that passed from our refractory time

        if self.refractory_time < 0: # If the neuron is not in its refractory period
            #########################
            # TODO: Integrate the input current
            self.v = None
            #########################
        
        if self.v >= self.v_th: # If the potential reaches the threshold
            #########################
            # TODO: Implement the firing behavior
            self.refractory_time = None  # Enter the refractory period
            self.output = None           # Emit a spike
            self.v = 0                          # Reset the potential
            #########################
        else: # If the potential is below the threshold
            self.output = 0 # Do not fire
        
        return self.output

Then, if we plot the output of our neuron, we should see it spike when the voltage reaches the threshold, and then be silent for the refractory period.

In [ ]:
duration = 6  # Duration of the simulation
T_step = 0.001 # Time step size
times = np.arange(0, duration, T_step) # Create a range of time values

neuron = FirstOrderLIF(tau_ref = 0.2) # Create a new LIF neuron

def square_wave(t): # Our input current function will be a square wave
    return 0 if (t % 2) < 1 else 1.1

I_history = []
v_history = []
output_history = []
vth_history = []

for t in times: # Iterate over each time step
    I = square_wave(t)     # Get the input current at this time
    neuron.step(I, T_step) # Advance the neuron one time step

    I_history.append(I)    # Record the input current
    v_history.append(neuron.v) # Record the neuron's potential
    output_history.append(neuron.output * T_step / 10) # Record the neuron's output (scaled)
    vth_history.append(neuron.v_th) # Record the neuron's threshold


fig = plt.figure(figsize=(16, 4), dpi=120) # Wider time axis
ax = plt.gca()
plt.plot(times, I_history, color="grey", linestyle="--")
plt.plot(times, vth_history, color="green", linestyle="--")
plt.plot(times, v_history)
plt.plot(times, output_history, color="red", linewidth=2.5)
plt.xlabel('Time (s)') # Label the x-axis
plt.legend(['Input (normalized)', 'Threshold', 'Membrane potential (normalized)', 'Spikes (scaled)'], loc='upper right') # Add a legend
plt.ylabel('Normalized values / scaled spikes')
ax.set_xlim(0, duration)
ax.set_xticks(np.arange(0, duration + 0.1, 0.2)) # Labels every 0.2 s
ax.set_xticks(np.arange(0, duration + 0.05, 0.1), minor=True) # Minor ticks every 0.1 s
ax.tick_params(axis='x', labelsize=9)
ax.grid(which='major', alpha=0.3)
ax.grid(which='minor', axis='x', alpha=0.12)
plt.tight_layout()
plt.show() # Display the plot


# Expected output:
# You are expected to see a red curve with spikes at the bottom.

The next simulation is for optional parameter exploration. Try changing `v_th`, `tau_ref`, and `tau_rc` to see how they affect firing. **Use the first output figure above, with its default parameters, for Task 2.**

In [ ]:
# Simulation parameters
v_th    = 1.000 # Threshold voltage for neuron <-SLIDE(0.1 to 1 by 0.001)
tau_ref = 0.200 # Refractory period for neuron <-SLIDE(0 to 2 by 0.001)
tau_rc  = 0.10  # Time constant for neuron     <-SLIDE(0.01 to 1.2 by 0.01)

duration = 6   # Duration of the simulation
T_step = 0.001 # Time step size


times = np.arange(0, duration, T_step) # Create a range of time values
neuron = FirstOrderLIF(tau_rc=tau_rc, tau_ref=tau_ref, v_th=v_th) # Create a new LIF neuron

def square_wave(t): # Our input current function will be a square wave
    return 0 if (t % 2) < 1 else 1.1

I_history = []
v_history = []
output_history = []
vth_history = []

for t in times: # Iterate over each time step
    I = square_wave(t)     # Get the input current at this time
    neuron.step(I, T_step) # Advance the neuron one time step

    I_history.append(I)    # Record the input current
    v_history.append(neuron.v) # Record the neuron's potential
    output_history.append(neuron.output * T_step / 10) # Record the neuron's output (scaled)
    vth_history.append(neuron.v_th) # Record the neuron's threshold


fig = plt.figure(figsize=(16, 4), dpi=120) # Wider time axis
ax = plt.gca()
plt.plot(times, I_history, color="grey", linestyle="--")
plt.plot(times, vth_history, color="green", linestyle="--")
plt.plot(times, v_history)
plt.plot(times, output_history, color="red", linewidth=2.5)
plt.xlabel('Time (s)') # Label the x-axis
plt.legend(['Input (normalized)', 'Threshold', 'Membrane potential (normalized)', 'Spikes (scaled)'], loc='upper right') # Add a legend
plt.ylabel('Normalized values / scaled spikes')
ax.set_xlim(0, duration)
ax.set_xticks(np.arange(0, duration + 0.1, 0.2)) # Labels every 0.2 s
ax.set_xticks(np.arange(0, duration + 0.05, 0.1), minor=True) # Minor ticks every 0.1 s
ax.tick_params(axis='x', labelsize=9)
ax.grid(which='major', alpha=0.3)
ax.grid(which='minor', axis='x', alpha=0.12)
plt.tight_layout()
plt.show() # Display the plot

### Task 1B. From input current to firing rate

How does a stronger input change the neuron's firing, and how does the refractory period limit it? 

Before running the experiment, predict the effect of increasing each parameter.

Implement `measure_firing_rate` below. For each call:

1. Create a **fresh** neuron with the specified refractory period, `tau_rc=0.2`, and `v_th=1.0`. Do not reuse the neuron from the first figure.
2. Advance it through every time step with the constant input `I_amp`.
3. Count an event whenever the **raw return value** of `step` is positive. 
4. Return the spike count divided by the simulated duration, in spikes/s (Hz).

Use the complete 5 s observation window, including the initial charging period. 

Complete the function and the inner loop of the parameter sweep; plotting code is provided. The currents and voltages are normalized; time is in seconds.


In [ ]:
def measure_firing_rate(I_amp, tau_ref, duration=5.0, dt=0.001):
    n_steps = int(round(duration / dt))

    ### START YOUR CODE 
    # TODO: Create a fresh FirstOrderLIF neuron with the required parameters.
    neuron = None
    spike_count = 0

    # TODO: Simulate n_steps updates and count firing events.
    # Write a loop, call neuron.step, and use an if statement.


    # TODO: Convert the count into a rate using n_steps * dt, and return it
    return None

    raise NotImplementedError("Implement the simulation and spike counter")
    ### END YOUR CODE ###


In [ ]:
# Self-checks: run after completing measure_firing_rate.
assert measure_firing_rate(0.0, 0.2) == 0, "Zero input should not cause firing."
assert measure_firing_rate(0.8, 0.2) == 0, "Subthreshold input should not cause firing."
assert 0 < measure_firing_rate(2.0, 0.2) < measure_firing_rate(2.0, 0.0), \
    "Adding a refractory period should reduce firing at this input."
assert measure_firing_rate(2.0, 0.2) == measure_firing_rate(2.0, 0.2), \
    "Each call should start from a fresh neuron."
print("Basic checks passed; also inspect your curves.")


In [ ]:
input_levels = [0.0, 0.5, 0.8, 1.0, 1.1, 1.5, 2.0, 3.0, 5.0]
refractory_periods = [0.0, 0.2]
rates_by_ref = {}

for ref in refractory_periods:
    rates = []
    for current in input_levels:
        ### START YOUR CODE ###
        # TODO: Call your function with current and ref, then append its rate.
        raise NotImplementedError("Complete the parameter sweep")
        ### END YOUR CODE ###

    rates_by_ref[ref] = rates

# Plotting is provided.
fig, ax = plt.subplots(figsize=(7, 4))
for ref, rates in rates_by_ref.items():
    ax.plot(input_levels, rates, "o-", label=f"tau_ref = {ref} s")
ax.set(xlabel="Constant input (normalized)", ylabel="Mean firing rate (Hz)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### Task 1B response

In 2–3 sentences, compare your predictions with the curves. Explain (a) why some inputs produce no spikes and (b) why a longer refractory period reduces firing at the same suprathreshold input. Refer to at least one pair of measured rates.

*Write your prediction and explanation here.*


## Task 2. Read and explain the output figure

Use the **first output figure in Task 1**, with the default parameters (`v_th = 1.0`, `tau_ref = 0.2 s`, `tau_rc = 0.2 s`, `T_step = 0.001 s`). Examine the full time axis and answer the following questions in words. No additional code is required.

1. How many **integration processes (积分过程)** are shown in the figure? For this question, count each continuous rising segment of the membrane-potential trace under input as one integration process, including segments that do not reach the firing threshold. Briefly explain how you identified them.
2. How many times does the neuron **fire (神经元激发)**? Explain which feature of the figure you used to count firing events.
3. Which time intervals correspond to the **refractory period (不应期)**? Give the approximate start and end times of each interval in seconds and briefly explain your reasoning.

Submit the completed notebook with your Task 1A and 1B code, the first voltage/spike figure, the firing-rate comparison figure, and your written answers for Task 1B and Task 2. You may annotate the figure to support your explanation, but a separate zoomed figure is not required.


### Task 2 response

**1. Number of integration processes and explanation:**

*Write your answer here.*

**2. Number of firing events and explanation:**

*Write your answer here.*

**3. Refractory intervals and explanation:**

| Interval | Start time (s) | End time (s) |
| --- | --- | --- |
| R1 | | |

*Add a row for each refractory interval and explain your reasoning below.*
